In [10]:
import json
from pathlib import Path

input_path = Path(
    "../src/exp_results/eval_dpo_base_qwen18_hh_1202_ep2/prob_test_gen_response.jsonl"
)
output_path = Path(
    "../src/exp_results/eval_dpo_base_qwen18_hh_1202_ep2/rm_input.jsonl"
)

def extract_last_answer(text: str) -> str:
    idx = text.rfind("Assistant:")
    if idx == -1:
        return text.strip()
    return text[idx + len("Assistant:"):].strip()

def clean_answer(prompt: str, response: str) -> str:
    if response.startswith(prompt):
        response = response[len(prompt):]
    return extract_last_answer(response)

with input_path.open(encoding='utf-8') as fin, output_path.open("w",encoding='utf-8') as fout:
    for line in fin:
        item = json.loads(line)

        prompt = item["prompt"].strip()
        response = item["response"]

        # 去掉 prompt 中的 Assistant:
        context = prompt.replace("\n\nAssistant:", "").strip()
        answer = clean_answer(prompt, response)

        fout.write(json.dumps({
            "context": context,
            "answer": answer
        }, ensure_ascii=False) + "\n")

print(f"Saved RM input to {output_path}")

Saved RM input to ..\src\exp_results\eval_dpo_base_qwen18_hh_1202_ep2\rm_input.jsonl


In [ ]:
import json
from pathlib import Path

input_path = Path(
    "src/exp_results/eval_dpo_base_qwen18_hh_1202_ep2/rm_input.jsonl"
)
output_path = Path(
    "src/exp_results/eval_dpo_base_qwen18_hh_1202_ep2/rm_scored.jsonl"
)

with input_path.open(encoding="utf-8") as fin, \
     output_path.open("w", encoding="utf-8") as fout:

    for line in fin:
        item = json.loads(line)

        rm_input = f"Human: {item['context']}\nAssistant: {item['answer']}"

        inputs = tokenizer(
            rm_input,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(model.device)

        with torch.no_grad():
            reward = model(**inputs).logits.squeeze().item()

        fout.write(json.dumps({
            **item,
            "reward": reward
        }, ensure_ascii=False) + "\n")
